# NeRF 01 -- Volume Rendering: The Math That Makes Neural Radiance Fields Work

> **Geo-Instructor · NeRF/3DGS Career Track · Notebook 1 of 3**

---

A NeRF is just a **neural function** that maps (x, y, z, direction) -> (color, density).
But to turn those values into a pixel, you need **volume rendering** -- the integral
that accumulates color along a ray through the volume.

This notebook derives and implements that integral from scratch.

```
Runtime: ~5 min  |  No GPU  |  numpy + scipy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
np.random.seed(7)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace','axes.grid':True,'grid.alpha':0.3})
print('Ready.')

---
## Part 1 -- The Volume Rendering Equation

Imagine a ray entering a participating medium (smoke, fog, a neural volume).
At each point along the ray, particles can emit light AND absorb/occlude it.

The **emission-absorption** volume rendering equation:
```
  C(r) = integral_t_n^t_f  T(t) * sigma(t) * c(t)  dt

  T(t) = exp( -integral_t_n^t  sigma(s) ds )   <- transmittance

  sigma(t) = density at t      (how opaque)
  c(t)     = color at t        (what color it emits)
  T(t)     = how much light reaches t from camera (starts at 1, decays)
```

In discrete form (N samples along ray):
```
  C = sum_i  T_i * (1 - exp(-sigma_i * delta_i)) * c_i
  T_i = exp( -sum_{j<i} sigma_j * delta_j )
  delta_i = t_{i+1} - t_i    <- distance between samples
```

In [ ]:
# Visualize transmittance for different density profiles
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Transmittance T(t) along a ray -- how much light survives', fontsize=11)

t = np.linspace(0, 5, 500)
profiles = [
    ('Uniform density', np.full_like(t, 0.5)),
    ('Surface at t=2', np.exp(-0.5*(t-2)**2) * 5),
    ('Two surfaces', np.exp(-2*(t-1.5)**2)*4 + np.exp(-2*(t-3.5)**2)*4),
]

for ax, (title, sigma) in zip(axes, profiles):
    dt = t[1] - t[0]
    # Transmittance: T(t) = exp(-cumulative_sigma * dt)
    T = np.exp(-np.cumsum(sigma) * dt)
    T = np.clip(T, 0, 1)
    # Weights: w_i = T_i * (1 - exp(-sigma_i * dt))
    alpha = 1 - np.exp(-sigma * dt)
    weights = T * alpha

    ax2 = ax.twinx()
    ax.fill_between(t, sigma/sigma.max(), alpha=0.25, color='steelblue', label='sigma (normalized)')
    ax.plot(t, sigma/sigma.max(), 'steelblue', lw=1.5)
    ax2.plot(t, T, 'tomato', lw=2, label='T(t) transmittance')
    ax2.plot(t, weights/weights.max(), 'seagreen', lw=2, ls='--', label='weights (normalized)')
    ax.set_xlabel('t (ray depth)'); ax.set_ylabel('density', color='steelblue')
    ax2.set_ylabel('T(t)', color='tomato'); ax2.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=9)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=7, loc='upper right')

plt.tight_layout(); plt.show()
print('Weights concentrate near surfaces -- that is where the rendering signal is.')

In [ ]:
def volume_render(sigmas, colors, deltas):
    """
    Discrete volume rendering integral.
    sigmas: (N,) density per sample
    colors: (N, 3) RGB per sample
    deltas: (N,) distance between consecutive samples
    Returns: rendered_color (3,), weights (N,), T_vals (N,)
    """
    alphas = 1 - np.exp(-sigmas * deltas)
    T = np.ones(len(sigmas))
    for i in range(1, len(sigmas)):
        T[i] = T[i-1] * (1 - alphas[i-1])
    weights = T * alphas
    color = (weights[:, None] * colors).sum(axis=0)
    return color, weights, T

# Demo: render a ray through a red sphere and blue background
N = 128
t_near, t_far = 0.0, 6.0
t_vals = np.linspace(t_near, t_far, N)
deltas = np.full(N, (t_far - t_near) / N)

# Scene: red sphere at t in [2, 3], blue background
sigma = np.zeros(N)
sigma += np.exp(-30*(t_vals - 2.5)**2) * 20   # sphere
bg_sigma = 0.02

colors = np.zeros((N, 3))
colors[:, 0] = np.clip(np.exp(-15*(t_vals-2.5)**2), 0, 1)  # red sphere
colors[:, 2] = np.clip(1 - np.exp(-15*(t_vals-2.5)**2), 0, 1) * 0.6  # blue bg

pixel_color, weights, T_vals = volume_render(sigma, colors, deltas)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(t_vals, sigma, 'steelblue', lw=2); axes[0].set_title('Density sigma(t)'); axes[0].set_xlabel('t')
axes[1].plot(t_vals, T_vals, 'tomato', lw=2, label='T(t)'); axes[1].plot(t_vals, weights, 'seagreen', lw=2, label='weights'); axes[1].legend(); axes[1].set_title('Transmittance & weights'); axes[1].set_xlabel('t')
color_display = np.ones((50, 200, 3))
color_display[:] = np.clip(pixel_color, 0, 1)
axes[2].imshow(color_display); axes[2].set_title(f'Rendered pixel: RGB=[{pixel_color[0]:.2f},{pixel_color[1]:.2f},{pixel_color[2]:.2f}]'); axes[2].axis('off')
plt.tight_layout(); plt.show()

---
## Part 2 -- Stratified & Importance Sampling

Uniform sampling wastes compute on empty space.
**Stratified sampling** divides the ray into N bins and samples one point per bin.
**Importance sampling** then concentrates more samples near surfaces (high weight regions).

In [ ]:
def stratified_sample(t_near, t_far, N, perturb=True):
    """Sample N points: one per stratum, optionally jittered."""
    t_edges = np.linspace(t_near, t_far, N+1)
    if perturb:
        return t_edges[:-1] + np.random.uniform(0, (t_far-t_near)/N, N)
    return (t_edges[:-1] + t_edges[1:]) / 2

def importance_sample(t_coarse, weights_coarse, N_fine):
    """Sample N_fine points proportional to coarse weights."""
    weights = weights_coarse + 1e-5  # prevent division by zero
    pdf = weights / weights.sum()
    cdf = np.cumsum(pdf)
    u = np.random.uniform(0, 1, N_fine)
    # Inverse CDF sampling
    t_fine = np.interp(u, cdf, t_coarse)
    return np.sort(t_fine)

# Coarse pass
t_coarse = stratified_sample(0, 6, 64)
deltas_c = np.diff(np.concatenate([t_coarse, [6.0]]))
sigma_c = np.exp(-30*(t_coarse-2.5)**2)*20
colors_c = np.zeros((64, 3)); colors_c[:,0] = np.exp(-15*(t_coarse-2.5)**2)
_, weights_c, _ = volume_render(sigma_c, colors_c, deltas_c)

# Fine pass
t_fine = importance_sample(t_coarse, weights_c, 128)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (ts, title) in zip(axes, [
    (t_coarse, f'Stratified coarse ({len(t_coarse)} samples)'),
    (np.sort(np.concatenate([t_coarse, t_fine])), f'+ Importance fine (total {len(t_coarse)+len(t_fine)} samples)')]):
    ax.scatter(ts, np.zeros_like(ts) + np.random.uniform(-0.1,0.1,len(ts)), s=8, alpha=0.5)
    ax.axvspan(2.0, 3.0, alpha=0.15, color='tomato', label='Surface region')
    ax.set_xlim(0, 6); ax.set_ylim(-0.5, 0.5)
    ax.set_title(title); ax.set_xlabel('t'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Importance sampling concentrates samples where weights are high (near surfaces).')

---
## Part 3 -- Tiny 2D Neural Field

A NeRF is just an MLP that overfits to a scene. Before 3D, let's build a
**2D neural field** that maps (x, y) -> color, fitting a small image.
All backprop implemented manually with numpy.

In [ ]:
def positional_encoding(x, n_freqs=6):
    """Fourier feature positional encoding. x: (..., D) -> (..., D*(1+2*n_freqs))"""
    freqs = 2.0 ** np.arange(n_freqs)  # [1, 2, 4, 8, 16, 32]
    x_enc = [x]
    for freq in freqs:
        x_enc.append(np.sin(freq * np.pi * x))
        x_enc.append(np.cos(freq * np.pi * x))
    return np.concatenate(x_enc, axis=-1)

# Show why positional encoding helps: high-freq detail
ts = np.linspace(-1, 1, 500)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, n in zip(axes, [0, 3, 6]):
    if n == 0:
        feat = ts.reshape(-1,1)
    else:
        feat = positional_encoding(ts.reshape(-1,1), n_freqs=n)
    ax.set_title(f'n_freqs={n}  -> dim={feat.shape[1]}', fontsize=9)
    for i in range(min(6, feat.shape[1])):
        ax.plot(ts, feat[:,i], alpha=0.6, lw=1)
    ax.set_xlabel('t')
plt.suptitle('Positional encoding: more frequencies = richer input representation', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# 2D Neural field: fit a target image
target_size = 32
xx, yy = np.meshgrid(np.linspace(-1,1,target_size), np.linspace(-1,1,target_size))
# Target: concentric rings
r = np.sqrt(xx**2 + yy**2)
target = (np.sin(r * 8) + 1) / 2

# Encode input coordinates
coords = np.column_stack([xx.ravel(), yy.ravel()])  # (N,2)
coords_enc = positional_encoding(coords, n_freqs=5)  # (N, 2*11=22)
IN = coords_enc.shape[1]; OUT = 1

# Tiny MLP: IN -> 64 -> 64 -> OUT
np.random.seed(42)
W1 = np.random.randn(IN, 64) * 0.1
b1 = np.zeros(64)
W2 = np.random.randn(64, 64) * 0.1
b2 = np.zeros(64)
W3 = np.random.randn(64, OUT) * 0.1
b3 = np.zeros(OUT)

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -50, 50)))
def relu(x): return np.maximum(0, x)

def forward(x, W1,b1,W2,b2,W3,b3):
    h1 = relu(x @ W1 + b1)
    h2 = relu(h1 @ W2 + b2)
    out = sigmoid(h2 @ W3 + b3)
    return out, h1, h2

# Mini-batch SGD
lr = 0.01; n_iters = 300; batch = 256
loss_history = []
for it in range(n_iters):
    idx = np.random.choice(len(coords_enc), batch, replace=False)
    x_b = coords_enc[idx]; y_b = target.ravel()[idx:, None] if False else target.ravel()[idx].reshape(-1,1)
    pred, h1, h2 = forward(x_b, W1,b1,W2,b2,W3,b3)
    loss = ((pred - y_b)**2).mean()
    loss_history.append(loss)
    # Backprop
    dL = 2*(pred - y_b) / batch
    dW3 = h2.T @ dL; db3 = dL.sum(axis=0)
    dh2 = dL @ W3.T * (h2 > 0)
    dW2 = h1.T @ dh2; db2 = dh2.sum(axis=0)
    dh1 = dh2 @ W2.T * (h1 > 0)
    dW1 = x_b.T @ dh1; db1 = dh1.sum(axis=0)
    W3-=lr*dW3; b3-=lr*db3; W2-=lr*dW2; b2-=lr*db2; W1-=lr*dW1; b1-=lr*db1

final_pred, _, _ = forward(coords_enc, W1,b1,W2,b2,W3,b3)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(target, cmap='viridis', vmin=0, vmax=1); axes[0].set_title('Target'); axes[0].axis('off')
axes[1].imshow(final_pred.reshape(target_size, target_size), cmap='viridis', vmin=0, vmax=1); axes[1].set_title(f'MLP after {n_iters} iters'); axes[1].axis('off')
axes[2].semilogy(loss_history, 'steelblue', lw=1.5); axes[2].set_title('Training loss'); axes[2].set_xlabel('Iteration')
plt.tight_layout(); plt.show()
print(f'Final MSE: {loss_history[-1]:.5f}')

---
## Exercises

### Exercise 1 -- PSNR metric
Implement `psnr(pred, target) = -10 * log10(mse)`. Higher is better. A perfect reconstruction = infinity. Typical NeRF achieves 25-35 dB.

### Exercise 2 -- Without positional encoding
Train the same 2D field without positional encoding (just pass raw xy coordinates). Compare the reconstruction. The MLP will struggle with high-frequency detail.

### Exercise 3 -- 3D extension
Extend volume rendering to render a full 2D image: cast one ray per pixel, accumulate color. This is the core NeRF rendering loop.

---
## Summary: NeRF Evolution

| Method | Key Idea | Speed | Quality |
|--------|----------|-------|--------|
| NeRF (2020) | MLP + positional encoding | Very slow | High |
| Instant-NGP (2022) | Hash grid + tiny MLP | Fast | High |
| 3DGS (2023) | Gaussian primitives, no MLP | Real-time | Very High |
| Nerfstudio | Framework for all of above | Varies | Varies |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  Volume rendering integral -> the core math of NeRF')
print('  Transmittance T(t)        -> how much light survives to depth t')
print('  Importance sampling       -> concentrate compute near surfaces')
print('  Positional encoding       -> let MLP represent high frequencies')
print('  Neural field              -> overfits an MLP to a scene')
print()
print('Next: 3D Gaussian Splatting -- replace the MLP with explicit Gaussians')